# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display summary
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Retrieve record sets and their fields using @id

print("Available Record Sets:")
for rs in dataset.metadata.record_sets:
    print(f"- RecordSet @id: {rs['@id']}, name: {rs.get('name','<no name>')}")
    if 'fields' in rs:
        for f in rs['fields']:
            print(f"    - Field @id: {f['@id']}  --  name: {f.get('name','<no name>')} (dataType: {f.get('dataType','<unknown>')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets discovered.
record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    dataframes[record_set] = pd.DataFrame(records)

# Choose the primary record set for demonstration (likely the main data table)
if record_sets:
    primary_record_set = record_sets[0]
    print(f"Fields available in record set {primary_record_set}:")
    print(dataframes[primary_record_set].columns.tolist())
    display(dataframes[primary_record_set].head())
else:
    print("No record sets available in this dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example: Use molecular weight (MW) if present for filtering/normalization
import numpy as np

# Some likely field IDs from overview step (update as appropriate):
# Let's search for numeric fields in the dataframe
numeric_candidates = [c for c in dataframes[primary_record_set].columns if dataframes[primary_record_set][c].apply(lambda x: isinstance(x, (int, float, np.number))).any()]
if not numeric_candidates:
    # Try to auto-convert numeric-like columns
    for c in dataframes[primary_record_set].columns:
        try:
            dataframes[primary_record_set][c] = pd.to_numeric(dataframes[primary_record_set][c])
        except Exception:
            continue
    numeric_candidates = [c for c in dataframes[primary_record_set].columns if pd.api.types.is_numeric_dtype(dataframes[primary_record_set][c])]

if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Using {numeric_field} as numeric field for filtering & normalization.")
    threshold = dataframes[primary_record_set][numeric_field].mean()
    filtered_df = dataframes[primary_record_set][dataframes[primary_record_set][numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try group by a categorical field
    group_candidates = [c for c in dataframes[primary_record_set].columns if c != numeric_field and pd.api.types.is_object_dtype(dataframes[primary_record_set][c])]
    if group_candidates:
        group_field = group_candidates[0]
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())
else:
    print("No numeric fields detected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if we have a numeric field
if numeric_candidates:
    plt.figure(figsize=(8,5))
    sns.histplot(dataframes[primary_record_set][numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # If we also have a group field, show boxplot
    if group_candidates:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=dataframes[primary_record_set])
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print("No suitable numeric fields for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* The dataset provides a rich table of protein-related features from mass spectrometry on human mast cell extracellular vesicles.
* We demonstrated dynamically loading fields and record sets via their `@id` and automatically explored a primary numeric field, showing its distribution and summary statistics.
* The `mlcroissant` library allows robust access to complex FAIR datasets specified with Croissant schemas, enabling scalable and reproducible scientific data analysis.